In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [3]:
files = os.listdir("/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET/Dataset_Li-ion")
print(files)

['utils.py', 'Readme file - Description of Experimental Tests.txt', 'Technical Information and Experimental Test Results for LG 18650HG2.pdf', '10degC', '0degC', '25degC', '40degC', 'n10degC', 'n20degC']


In [4]:
import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
warnings.filterwarnings("ignore")

In [5]:
try:
    from xgboost import XGBRegressor
except ImportError:
    !pip -q install xgboost
    from xgboost import XGBRegressor

In [6]:
PROJECT_PATH = "/content/drive/MyDrive/BATTERY_SOC_PROJECT"
OUTPUT_DIR = os.path.join(PROJECT_PATH, "day19_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Day 19 output folder:")
print(OUTPUT_DIR)

Day 19 output folder:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day19_outputs


In [7]:
def find_file(filename):
    matches = []
    for root, dirs, files in os.walk(PROJECT_PATH):
        if filename in files:
            matches.append(os.path.join(root, filename))
    if not matches:
        return None
    return matches[0]

In [8]:
feature_file = find_file("final_feature_list_day13.json")
if feature_file is None:
    raise FileNotFoundError("final_feature_list_day13.json was not found.")
with open(feature_file, "r") as f:
    feature_data = json.load(f)
if isinstance(feature_data, list):
    FEATURES = feature_data
elif isinstance(feature_data, dict):
    possible_keys = ["features","feature_list","selected_features","final_features","input_features"]
    FEATURES = None
    for key in possible_keys:
        if key in feature_data and isinstance(feature_data[key], list):
            FEATURES = feature_data[key]
            break
    if FEATURES is None:
        for value in feature_data.values():
            if isinstance(value, list) and all(
                isinstance(x, str) for x in value
            ):
                FEATURES = value
                break
    if FEATURES is None:
        raise ValueError( "Could not identify the feature list inside final_feature_list_day13.json")
else:
    raise ValueError("Unexpected feature-list JSON format.")
print("\nNumber of Day 13 features:", len(FEATURES))
print("Features:")
print(FEATURES)


Number of Day 13 features: 7
Features:
['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'WhAccu', 'Power']


In [9]:
SOC_TRAIN_FILE = find_file("SOC_train_engineered_day13.csv")
SOC_VAL_FILE = find_file("SOC_validation_engineered_day13.csv")
TIME_TRAIN_FILE = find_file("remaining_time_train_engineered_day13.csv")
TIME_VAL_FILE = find_file("remaining_time_validation_engineered_day13.csv")
required_files = {"SOC train": SOC_TRAIN_FILE,"SOC validation": SOC_VAL_FILE,"Remaining Time train": TIME_TRAIN_FILE,"Remaining Time validation": TIME_VAL_FILE}
for name, path in required_files.items():
    if path is None:
        raise FileNotFoundError(f"{name} file was not found.")

In [10]:
soc_train = pd.read_csv(SOC_TRAIN_FILE)
soc_val = pd.read_csv(SOC_VAL_FILE)
time_train = pd.read_csv(TIME_TRAIN_FILE)
time_val = pd.read_csv(TIME_VAL_FILE)
print("\nLoaded Day 13 frozen split.")
print("SOC train:", soc_train.shape)
print("SOC validation:", soc_val.shape)
print("Remaining Time train:", time_train.shape)
print("Remaining Time validation:", time_val.shape)


Loaded Day 13 frozen split.
SOC train: (9926, 10)
SOC validation: (7376, 10)
Remaining Time train: (41015, 11)
Remaining Time validation: (1349, 11)


In [11]:
def prepare_data(train_df, val_df, target):
    available_features = [
        f for f in FEATURES
        if f in train_df.columns and f in val_df.columns]
    if len(available_features) == 0:
        raise ValueError(f"No selected Day 13 features were found for {target}.")
    X_train = train_df[available_features].copy()
    X_val = val_df[available_features].copy()
    y_train = pd.to_numeric(train_df[target], errors="coerce")
    y_val = pd.to_numeric(val_df[target], errors="coerce")
    for col in available_features:
        X_train[col] = pd.to_numeric( X_train[col], errors="coerce")
        X_val[col] = pd.to_numeric( X_val[col], errors="coerce")
    train_medians = X_train.median()
    X_train = X_train.fillna(train_medians)
    X_val = X_val.fillna(train_medians)
    train_mask = y_train.notna()
    val_mask = y_val.notna()
    X_train = X_train.loc[train_mask].reset_index(drop=True)
    y_train = y_train.loc[train_mask].reset_index(drop=True)
    X_val = X_val.loc[val_mask].reset_index(drop=True)
    y_val = y_val.loc[val_mask].reset_index(drop=True)
    return (X_train,X_val,y_train,y_val,available_features)
soc_X_train, soc_X_val, soc_y_train, soc_y_val, soc_features = prepare_data(soc_train,soc_val,"SOC")
time_X_train, time_X_val, time_y_train, time_y_val, time_features = prepare_data(time_train,time_val,"Remaining Charging Time")
print("\nSOC features:", soc_features)
print("Remaining Time features:", time_features)


SOC features: ['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'WhAccu', 'Power']
Remaining Time features: ['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'WhAccu', 'Power']


In [12]:
def calculate_metrics(model, X, y, dataset_name):
    pred = model.predict(X)
    mae = mean_absolute_error(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    r2 = r2_score(y, pred)
    return {"Dataset": dataset_name,"MAE": mae,"RMSE": rmse,"R2": r2}


In [13]:
search_strategy = {
    "purpose": "Controlled hyperparameter/parameter experiments",
    "data_split": "Frozen Day 13 train/validation/test split",
    "model_selection_data": "Validation set only",
    "test_set_usage": "Test set not loaded or used for model selection",
    "primary_selection_metric": "Validation MAE",
    "secondary_selection_metric": "Validation RMSE",
    "additional_metric": "Validation R2",
    "Random_Forest": {"parameters_changed": ["n_estimators","max_depth","min_samples_split","min_samples_leaf"],"fixed_random_state": 42},
    "XGBoost": {"parameters_changed": ["n_estimators","max_depth","learning_rate","min_child_weight"],"other_parameters_fixed": True,"fixed_random_state": 42},
    "baseline_reference_models": ["Day 15 Linear Regression","Day 17 MLP"]}
with open(
    os.path.join(OUTPUT_DIR, "day19_search_strategy.json"),
    "w"
) as f:
    json.dump(search_strategy, f, indent=4)

In [14]:
rf_configs = {
    "RF_1_Baseline_Tuned": {"n_estimators": 250,"max_depth": 8,"min_samples_split": 10,"min_samples_leaf": 4},
    "RF_2_More_Regularization": {"n_estimators": 250,"max_depth": 6,"min_samples_split": 10,"min_samples_leaf": 5},
    "RF_3_More_Trees": {"n_estimators": 400,"max_depth": 8,"min_samples_split": 10,"min_samples_leaf": 4}
}

In [15]:
xgb_configs = {
    "XGB_1_Baseline_Tuned": {"n_estimators": 300,"max_depth": 3,"learning_rate": 0.03,"min_child_weight": 4},
    "XGB_2_More_Regularization": {"n_estimators": 300,"max_depth": 2,"learning_rate": 0.03,"min_child_weight": 5},
    "XGB_3_Slower_Learning": {"n_estimators": 400,"max_depth": 3,"learning_rate": 0.02,"min_child_weight": 4}
}

In [16]:
experiment_results = []
trained_experiments = {}
for config_name, params in rf_configs.items():
    print("\nRunning:", config_name)
    for target_name, X_train, X_val, y_train, y_val in [
        ( "SOC",soc_X_train,soc_X_val,soc_y_train,soc_y_val),

        ("Remaining Charging Time", time_X_train,time_X_val,time_y_train,time_y_val)
    ]:
        model = RandomForestRegressor(**params,random_state=42,n_jobs=-1)
        model.fit(X_train, y_train)
        train_metrics = calculate_metrics(model,X_train,y_train,"Train")
        val_metrics = calculate_metrics(model,X_val,y_val,"Validation")
        experiment_results.append({"Experiment": config_name,
            "Model": "Random Forest","Target": target_name,
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "min_samples_split": params["min_samples_split"],
            "min_samples_leaf": params["min_samples_leaf"],
            "Train_MAE": train_metrics["MAE"],
            "Train_RMSE": train_metrics["RMSE"],
            "Train_R2": train_metrics["R2"],
            "Validation_MAE": val_metrics["MAE"],
            "Validation_RMSE": val_metrics["RMSE"],
            "Validation_R2": val_metrics["R2"]
        })
        trained_experiments[(config_name, target_name)] = model
for config_name, params in xgb_configs.items():
    print("\nRunning:", config_name)
    for target_name, X_train, X_val, y_train, y_val in [
        ("SOC",soc_X_train,soc_X_val,soc_y_train,soc_y_val),
        ( "Remaining Charging Time",time_X_train,time_X_val,time_y_train,time_y_val)
    ]:
        model = XGBRegressor(**params,subsample=0.8,colsample_bytree=0.8,reg_alpha=0.1,reg_lambda=5.0,objective="reg:squarederror",random_state=42,n_jobs=-1)
        model.fit(X_train, y_train)
        train_metrics = calculate_metrics(model,X_train,y_train,"Train")
        val_metrics = calculate_metrics(model,X_val,y_val,"Validation")
        experiment_results.append({
            "Experiment": config_name,
            "Model": "XGBoost",
            "Target": target_name,
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "learning_rate": params["learning_rate"],
            "min_child_weight": params["min_child_weight"],
            "Train_MAE": train_metrics["MAE"],
            "Train_RMSE": train_metrics["RMSE"],
            "Train_R2": train_metrics["R2"],
            "Validation_MAE": val_metrics["MAE"],
            "Validation_RMSE": val_metrics["RMSE"],
            "Validation_R2": val_metrics["R2"]
        })
        trained_experiments[(config_name, target_name)] = model


Running: RF_1_Baseline_Tuned

Running: RF_2_More_Regularization

Running: RF_3_More_Trees

Running: XGB_1_Baseline_Tuned

Running: XGB_2_More_Regularization

Running: XGB_3_Slower_Learning


In [17]:
experiment_log = pd.DataFrame(experiment_results)
experiment_log = experiment_log.sort_values(by=["Target", "Validation_MAE", "Validation_RMSE"]).reset_index(drop=True)
experiment_log_path = os.path.join(OUTPUT_DIR,"day19_experiment_log.csv")
experiment_log.to_csv(experiment_log_path,index=False)
print("\n------ DAY 19 EXPERIMENT LOG ------n")
display(experiment_log)


------ DAY 19 EXPERIMENT LOG ------n


,Experiment,Model,Target,n_estimators,max_depth,min_samples_split,min_samples_leaf,Train_MAE,Train_RMSE,Train_R2,Validation_MAE,Validation_RMSE,Validation_R2,learning_rate,min_child_weight
0,RF_1_Baseline_Tuned,Random Forest,Remaining Charging Time,250,8,10.0,4.0,54.619269,259.753224,0.999612,66.009018,85.341677,0.996502,NaN,NaN
1,RF_3_More_Trees,Random Forest,Remaining Charging Time,400,8,10.0,4.0,54.445729,258.470984,0.999615,66.031102,85.453108,0.996493,NaN,NaN
2,XGB_1_Baseline_Tuned,XGBoost,Remaining Charging Time,300,3,NaN,NaN,129.596153,396.701981,0.999094,210.954248,316.450131,0.951900,0.03,4.0
3,XGB_3_Slower_Learning,XGBoost,Remaining Charging Time,400,3,NaN,NaN,138.774433,432.659125,0.998922,238.211358,357.287447,0.938685,0.02,4.0
4,RF_2_More_Regularization,Random Forest,Remaining Charging Time,250,6,10.0,5.0,153.296598,590.799033,0.997990,292.962323,359.473769,0.937932,NaN,NaN
5,XGB_2_More_Regularization,XGBoost,Remaining Charging Time,300,2,NaN,NaN,430.937382,1294.707691,0.990348,647.688946,1080.618637,0.439110,0.03,5.0
6,XGB_2_More_Regularization,XGBoost,SOC,300,2,NaN,NaN,1.250890,2.785292,0.995612,5.767199,8.348844,0.892573,0.03,5.0
7,XGB_3_Slower_Learning,XGBoost,SOC,400,3,NaN,NaN,0.537366,1.427497,0.998848,7.488319,9.850165,0.850464,0.02,4.0
8,XGB_1_Baseline_Tuned,XGBoost,SOC,300,3,NaN,NaN,0.503184,1.330074,0.998999,7.498428,9.944824,0.847576,0.03,4.0
9,RF_2_More_Regularization,Random Forest,SOC,250,6,10.0,5.0,0.654422,1.783248,0.998202,8.633458,11.365575,0.800913,NaN,NaN


In [18]:
best_configurations = []
for target in experiment_log["Target"].unique():
    target_results = experiment_log[experiment_log["Target"] == target].copy()
    target_results = target_results.sort_values(
        by=["Validation_MAE","Validation_RMSE","Validation_R2"],
        ascending=[True,True,False]
    )
    best_row = target_results.iloc[0]
    best_configurations.append(best_row)
best_validated_df = pd.DataFrame(best_configurations).reset_index(drop=True)
best_validated_path = os.path.join(OUTPUT_DIR,"day19_best_validated_configuration.csv")
best_validated_df.to_csv(best_validated_path,index=False)

print("\n------- BEST VALIDATED CONFIGURATION ------\n")

display(
    best_validated_df[
        ["Target","Model","Experiment","Validation_MAE","Validation_RMSE","Validation_R2"]]
)


------- BEST VALIDATED CONFIGURATION ------



,Target,Model,Experiment,Validation_MAE,Validation_RMSE,Validation_R2
0,Remaining Charging Time,Random Forest,RF_1_Baseline_Tuned,66.009018,85.341677,0.996502
1,SOC,XGBoost,XGB_2_More_Regularization,5.767199,8.348844,0.892573


In [19]:
best_json = {}
for _, row in best_validated_df.iterrows():
    target = row["Target"]
    config = {"Model": row["Model"],"Experiment": row["Experiment"],"Validation_MAE": float(row["Validation_MAE"]),"Validation_RMSE": float(row["Validation_RMSE"]),"Validation_R2": float(row["Validation_R2"])}
    best_json[target] = config
with open(
    os.path.join( OUTPUT_DIR,"day19_best_validated_configuration.json"),
    "w"
) as f:
    json.dump(best_json, f, indent=4)

In [20]:
best_models = {}

for _, row in best_validated_df.iterrows():
    target = row["Target"]
    model_name = row["Model"]
    experiment_name = row["Experiment"]
    model = trained_experiments[(experiment_name, target)]
    best_models[target] = model
    model_filename = (f"day19_best_{target.replace(' ', '_')}"f"_{model_name.replace(' ', '_')}.joblib")
    model_path = os.path.join(OUTPUT_DIR,model_filename)
    joblib.dump(
        {
            "model": model,
            "features": (soc_features
                if target == "SOC"
                else time_features
            ),
            "target": target,"selection_data": "Validation only","test_used_for_selection": False
        },
        model_path
    )
    print(f"Saved best {target} model:",model_path)

Saved best Remaining Charging Time model: /content/drive/MyDrive/BATTERY_SOC_PROJECT/day19_outputs/day19_best_Remaining_Charging_Time_Random_Forest.joblib
Saved best SOC model: /content/drive/MyDrive/BATTERY_SOC_PROJECT/day19_outputs/day19_best_SOC_XGBoost.joblib


In [21]:
overall_comparison = []

day15_results_file = find_file("day15_baseline_results.csv")
if day15_results_file is not None:
    day15_df = pd.read_csv(day15_results_file)
    for _, row in day15_df.iterrows():
        if str(row.get("Dataset", "")).lower() == "validation":
            overall_comparison.append({
                "Day": "Day 15",
                "Model": "Linear Regression",
                "Target": row.get("Target"),
                "Validation_MAE": row.get("MAE"),
                "Validation_RMSE": row.get("RMSE"),
                "Validation_R2": row.get("R2")
            })


day16_results_file = find_file("day16_tuned_model_results.csv")
if day16_results_file is not None:
    day16_df = pd.read_csv(day16_results_file)
    for _, row in day16_df.iterrows():
        if str(row.get("Dataset", "")).lower() == "validation":
            overall_comparison.append({
                "Day": "Day 16",
                "Model": row.get("Model"),
                "Target": row.get("Target"),
                "Validation_MAE": row.get("MAE"),
                "Validation_RMSE": row.get("RMSE"),
                "Validation_R2": row.get("R2")
            })

day17_results_file = find_file("day17_mlp_results.csv")
if day17_results_file is not None:
    day17_df = pd.read_csv(day17_results_file)
    for _, row in day17_df.iterrows():
        if str(row.get("Dataset", "")).lower() == "validation":
            overall_comparison.append({
                "Day": "Day 17",
                "Model": row.get("Model", "MLP"),
                "Target": row.get("Target"),
                "Validation_MAE": row.get("MAE"),
                "Validation_RMSE": row.get("RMSE"),
                "Validation_R2": row.get("R2")
            })

for _, row in best_validated_df.iterrows():
    overall_comparison.append({
        "Day": "Day 19",
        "Model": f"{row['Model']} - {row['Experiment']}",
        "Target": row["Target"],
        "Validation_MAE": row["Validation_MAE"],
        "Validation_RMSE": row["Validation_RMSE"],
        "Validation_R2": row["Validation_R2"]
    })

overall_comparison_df = pd.DataFrame(overall_comparison)

overall_comparison_path = os.path.join(OUTPUT_DIR,"day19_overall_model_comparison.csv")
overall_comparison_df.to_csv(overall_comparison_path,index=False)
print("\n-------OVERALL MODEL COMPARISON---------\n")
display(
    overall_comparison_df.sort_values(
        by=["Target","Validation_MAE"],
        ascending=[True,True]
    )
)


-------OVERALL MODEL COMPARISON---------



,Day,Model,Target,Validation_MAE,Validation_RMSE,Validation_R2
3,Day 16,Random Forest Tuned,Remaining Charging Time,66.009018,85.341677,0.996502
8,Day 19,Random Forest - RF_1_Baseline_Tuned,Remaining Charging Time,66.009018,85.341677,0.996502
7,Day 17,Small ANN / MLP,Remaining Charging Time,85.539953,117.117022,0.993412
5,Day 16,XGBoost Tuned,Remaining Charging Time,210.954248,316.450131,0.951900
1,Day 15,Linear Regression,Remaining Charging Time,3274.076803,3667.171231,-5.459444
9,Day 19,XGBoost - XGB_2_More_Regularization,SOC,5.767199,8.348844,0.892573
4,Day 16,XGBoost Tuned,SOC,7.498428,9.944824,0.847576
2,Day 16,Random Forest Tuned,SOC,9.072035,11.613402,0.792137
6,Day 17,Small ANN / MLP,SOC,11.614762,17.777087,0.512942
0,Day 15,Linear Regression,SOC,15.707497,21.039165,0.317793


In [22]:
findings = []
findings.append("Day 19 performed controlled hyperparameter experiments using the frozen Day 13 training and validation datasets.")
findings.append("The test set was not loaded or used for model selection.")
findings.append("Random Forest and XGBoost were evaluated using controlled changes to selected hyperparameters.")
findings.append("Validation MAE was used as the primary selection metric, with Validation RMSE as the secondary criterion and Validation R2 as an additional performance measure.")
findings.append("Linear Regression from Day 15 and MLP from Day 17 were included in the overall model comparison as reference candidate models."
)
for _, row in best_validated_df.iterrows():
    findings.append(
        f"For {row['Target']}, the best validated configuration "
        f"was {row['Model']} ({row['Experiment']}) with "
        f"Validation MAE={row['Validation_MAE']:.4f}, "
        f"Validation RMSE={row['Validation_RMSE']:.4f}, "
        f"and Validation R2={row['Validation_R2']:.4f}."
    )
findings_path = os.path.join(OUTPUT_DIR,"day19_initial_findings.txt")
with open(
    findings_path,
    "w"
) as f:
    for item in findings:
        f.write(item + "\n")

In [23]:
print("\nSaved outputs:")
for filename in sorted(
    os.listdir(OUTPUT_DIR)
):
    print(" -", filename)
print("\nTest set status:")
print("NOT USED for Day 19 model selection.")
print("\nBest validated configurations:")
display(
    best_validated_df[
        ["Target","Model","Experiment","Validation_MAE","Validation_RMSE","Validation_R2"]]
)


Saved outputs:
 - day19_best_Remaining_Charging_Time_Random_Forest.joblib
 - day19_best_SOC_XGBoost.joblib
 - day19_best_validated_configuration.csv
 - day19_best_validated_configuration.json
 - day19_experiment_log.csv
 - day19_initial_findings.txt
 - day19_overall_model_comparison.csv
 - day19_search_strategy.json

Test set status:
NOT USED for Day 19 model selection.

Best validated configurations:


,Target,Model,Experiment,Validation_MAE,Validation_RMSE,Validation_R2
0,Remaining Charging Time,Random Forest,RF_1_Baseline_Tuned,66.009018,85.341677,0.996502
1,SOC,XGBoost,XGB_2_More_Regularization,5.767199,8.348844,0.892573
